# 🧠 J.A.R.V.I.S. Mark II — Project A.R.C. (Autonomous Reflex Core)
## Sovereign SLM Distillation Notebook (100% Free Cloud GPU)

**Hardware Target**: Google Colab / Kaggle Free NVIDIA T4 GPU (16GB VRAM)
**Base Model**: `unsloth/Llama-3.2-3B-Instruct` (100% Western Foundation)
**Technique**: Unsloth 4-bit QLoRA (r=64, alpha=128) + FastLanguageModel
**Output**: 4-bit Quantized GGUF (`Q4_K_M`) for local Mac / PC execution

---
### Step 1: Check GPU Runtime

In [ ]:
!nvidia-smi

### Step 2: Install High-Speed Unsloth ML Engine
Unsloth achieves 5x faster training speeds and 80% lower VRAM usage.

In [ ]:
%%capture
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes

### Step 3: Load Western Foundation Model in 4-Bit NF4

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto-detects float16 for T4, bfloat16 for L4/A100
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Apply LoRA Adapters to Attention & FFN layers
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Llama 3.2 3B Model & LoRA Adapters Loaded Successfully!")

### Step 4: Load Master Training Dataset
Automatically downloads the clean 1,017-sample dataset from our sovereign cloud storage.

In [ ]:
import os
import urllib.request

DATASET_URL = "https://storage.googleapis.com/jarvis-vault-harshan-sovereign/distill/latest/train_master_arc.jsonl"
LOCAL_FILE = "train_master_arc.jsonl"

if not os.path.exists(LOCAL_FILE):
    print(f"⏳ Downloading sovereign dataset from {DATASET_URL}...")
    urllib.request.urlretrieve(DATASET_URL, LOCAL_FILE)
    print(f"✅ Dataset downloaded ({os.path.getsize(LOCAL_FILE):,} bytes)!")
else:
    print(f"✅ {LOCAL_FILE} already present ({os.path.getsize(LOCAL_FILE):,} bytes)!")

### Step 5: Format ChatML Dataset & Train Model

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

dataset = load_dataset("json", data_files={"train": "train_master_arc.jsonl"})

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

formatted_dataset = dataset["train"].map(formatting_prompts_func, batched = True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 300,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Commencing Distillation Training on Free GPU...")
trainer.train()
print("🎉 Training Complete!")

### Step 6: Test Reflex Inference & Sir's Persona

In [ ]:
FastLanguageModel.for_inference(model)
messages = [
    {"role": "system", "content": "You are J.A.R.V.I.S. Mark II, an autonomous sovereign cognitive exoskeleton for Sir (Harshan Sarvaiya)."},
    {"role": "user", "content": "Jarvis, report system readiness and review our distributed Kafka setup."}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True)
print(tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0])

### Step 7: Export to 4-Bit GGUF for Local Workstation & Mobile
Exports `Q4_K_M` GGUF ready to run on Apple Silicon Metal or Ollama.

In [ ]:
model.save_pretrained_gguf("jarvis_sovereign_1_gguf", tokenizer, quantization_method = "q4_k_m")
print("✅ Model exported to GGUF format! Check file browser on the left.")

### Step 8: Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r jarvis_sovereign_1_gguf /content/drive/MyDrive/
print("🏁 Sovereign model weights safely stored in your Google Drive!")